# Spotify Popularity Analysis

Exploring the relationship between song popularity, audio characteristics, genres, and release trends.

**Tools:** Polars · DuckDB · PostgreSQL · Quarto

In [1]:
import polars as pl

In [2]:
df= pl.read_csv("../data/spotify-tracks-dataset-detailed.csv")
df.head()

track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
str,str,str,str,i64,i64,bool,f64,f64,i64,f64,i64,f64,f64,f64,f64,f64,f64,i64,str
"""5SuOikwiRyPMVoIQDJUgSV""","""Gen Hoshino""","""Comedy""","""Comedy""",73,230666,false,0.676,0.461,1,-6.746,0,0.143,0.0322,0.000001,0.358,0.715,87.917,4,"""acoustic"""
"""4qPNDBW1i3p13qLCt0Ki3A""","""Ben Woodward""","""Ghost (Acoustic)""","""Ghost - Acoustic""",55,149610,false,0.42,0.166,1,-17.235,1,0.0763,0.924,0.000006,0.101,0.267,77.489,4,"""acoustic"""
"""1iJBSr7s7jYXzM8EGcbK5b""","""Ingrid Michaelson;ZAYN""","""To Begin Again""","""To Begin Again""",57,210826,false,0.438,0.359,0,-9.734,1,0.0557,0.21,0.0,0.117,0.12,76.332,4,"""acoustic"""
"""6lfxq3CG4xtTiEg7opyCyx""","""Kina Grannis""","""Crazy Rich Asians (Original Mo…","""Can't Help Falling In Love""",71,201933,false,0.266,0.0596,0,-18.515,1,0.0363,0.905,0.0000707,0.132,0.143,181.74,3,"""acoustic"""
"""5vjLSffimiIP26QG5WcN2K""","""Chord Overstreet""","""Hold On""","""Hold On""",82,198853,false,0.618,0.443,2,-9.681,1,0.0526,0.469,0.0,0.0829,0.167,119.949,4,"""acoustic"""


In [3]:
df.shape

(114000, 20)

In [4]:
df.tail()

track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
str,str,str,str,i64,i64,bool,f64,f64,i64,f64,i64,f64,f64,f64,f64,f64,f64,i64,str
"""2C3TZjDRiAzdyViavDJ217""","""Rainy Lullaby""","""#mindfulness - Soft Rain for M…","""Sleep My Little Boy""",21,384999,false,0.172,0.235,5,-16.393,1,0.0422,0.64,0.928,0.0863,0.0339,125.995,5,"""world-music"""
"""1hIz5L4IB9hN3WRYPOCGPw""","""Rainy Lullaby""","""#mindfulness - Soft Rain for M…","""Water Into Light""",22,385000,false,0.174,0.117,0,-18.318,0,0.0401,0.994,0.976,0.105,0.035,85.239,4,"""world-music"""
"""6x8ZfSoqDjuNa5SVP5QjvX""","""Cesária Evora""","""Best Of""","""Miss Perfumado""",22,271466,false,0.629,0.329,0,-10.895,0,0.042,0.867,0.0,0.0839,0.743,132.378,4,"""world-music"""
"""2e6sXL2bYv4bSz6VTdnfLs""","""Michael W. Smith""","""Change Your World""","""Friends""",41,283893,false,0.587,0.506,7,-10.889,1,0.0297,0.381,0.0,0.27,0.413,135.96,4,"""world-music"""
"""2hETkH7cOfqmz3LqZDHZf5""","""Cesária Evora""","""Miss Perfumado""","""Barbincor""",22,241826,false,0.526,0.487,1,-10.204,0,0.0725,0.681,0.0,0.0893,0.708,79.198,4,"""world-music"""


In [5]:
df["track_genre"].n_unique()

114

In [6]:
df["track_genre"].value_counts()

track_genre,count
str,u32
"""hard-rock""",1000
"""cantopop""",1000
"""gospel""",1000
"""chill""",1000
"""opera""",1000
…,…
"""pop-film""",1000
"""pagode""",1000
"""bluegrass""",1000


### Dataset Structure

The dataset contains 114,000 tracks across 114 genres, with exactly 1,000 tracks per genre. The records are grouped by genre rather than randomly ordered.

This balanced structure makes comparisons between genres more consistent, but the row order should not be treated as random.

In [7]:
stats = [
    pl.col("popularity").mean().alias("mean"),
    pl.col("popularity").median().alias("median"),
    pl.col("popularity").std().alias("std"),
    pl.col("popularity").min().alias("min"),
    pl.col("popularity").max().alias("max")
]
df.select(stats)

mean,median,std,min,max
f64,f64,f64,i64,i64
33.238535,35.0,22.305078,0,100


In [8]:
popularity_bins = (
    df
    .with_columns(
        pl.col("popularity")
        .cut(
            breaks=[20, 40, 60, 80],
            labels=["0-20", "21-40", "41-60", "61-80", "81-100"]
        )
        .alias("popularity_group")
    )
    .group_by("popularity_group")
    .agg(
        pl.len().alias("tracks")
    )
    .sort("popularity_group")
)

popularity_bins

popularity_group,tracks
enum,u32
"""0-20""",34177
"""21-40""",33149
"""41-60""",33104
"""61-80""",12616
"""81-100""",954


### Popularity Distribution

Popularity is concentrated toward the lower and middle ranges. Nearly 90% of tracks have a popularity score of 60 or below, while only a small fraction reach 81–100.

This suggests that highly popular tracks are relatively rare in the dataset, so popularity thresholds should be chosen carefully in later comparisons.

In [9]:
genre_popularity = (
    df
    .group_by("track_genre")
    .agg(
        pl.col("popularity").mean().alias("mean_popularity"),
        pl.col("popularity").median().alias("median_popularity"),
        pl.col("popularity").min().alias("min_popularity"),
        pl.col("popularity").max().alias("max_popularity")
    )
    .sort("mean_popularity", descending=True)
)

genre_popularity

track_genre,mean_popularity,median_popularity,min_popularity,max_popularity
str,f64,f64,i64,i64
"""pop-film""",59.283,60.0,0,80
"""k-pop""",56.896,60.0,0,88
"""chill""",53.651,57.0,0,93
"""sad""",52.379,54.0,0,83
"""grunge""",49.594,55.0,0,85
…,…,…,…,…
"""chicago-house""",12.339,10.0,0,78
"""detroit-techno""",11.174,8.0,0,58
"""latin""",8.297,0.0,0,98


In [10]:
zero_popularity = (
    df
    .group_by("track_genre")
    .agg(
        pl.col("popularity").eq(0).sum().alias("zero_popularity_tracks"),
        pl.len().alias("total_tracks")
    )
    .with_columns(
        (pl.col("zero_popularity_tracks") / pl.col("total_tracks") * 100)
        .alias("zero_popularity_pct")
    )
    .sort("zero_popularity_pct", descending=True)
)

zero_popularity

track_genre,zero_popularity_tracks,total_tracks,zero_popularity_pct
str,u32,u32,f64
"""jazz""",681,1000,68.1
"""iranian""",656,1000,65.6
"""romance""",636,1000,63.6
"""soul""",611,1000,61.1
"""latin""",588,1000,58.8
…,…,…,…
"""sertanejo""",1,1000,0.1
"""club""",1,1000,0.1
"""comedy""",0,1000,0.0


In [11]:
genre_analysis = (
    genre_popularity
    .join(
        zero_popularity.select(
            "track_genre",
            "zero_popularity_pct"
        ),
        on="track_genre"
    )
)
genre_analysis

track_genre,mean_popularity,median_popularity,min_popularity,max_popularity,zero_popularity_pct
str,f64,f64,i64,i64,f64
"""jazz""",13.628,0.0,0,83,68.1
"""iranian""",2.21,0.0,0,33,65.6
"""romance""",3.245,0.0,0,35,63.6
"""soul""",19.795,0.0,0,88,61.1
"""latin""",8.297,0.0,0,98,58.8
…,…,…,…,…,…
"""sertanejo""",47.866,47.0,0,63,0.1
"""club""",33.338,37.0,0,85,0.1
"""comedy""",24.628,23.0,19,75,0.0


In [12]:
genre_analysis.select(
    pl.corr(
        "zero_popularity_pct",
        "mean_popularity"
    ).alias("correlation")
)

correlation
f64
-0.428808


### Popularity and Zero-Score Tracks

Genre-level analysis shows a moderate negative correlation (-0.429) between the percentage of zero-popularity tracks and mean popularity. Genres containing more zero-score tracks generally have lower average popularity.

This suggests that the prevalence of zero-popularity tracks contributes to differences in genre-level popularity, so mean popularity should not be interpreted in isolation.

In [13]:
genre_analysis = genre_analysis.with_columns(
    (pl.col("mean_popularity") - pl.col("median_popularity"))
    .abs()
    .alias("mean_median_gap")
)
genre_analysis.sort(
    "mean_median_gap",
    descending=True
).head(10)

track_genre,mean_popularity,median_popularity,min_popularity,max_popularity,zero_popularity_pct,mean_median_gap
str,f64,f64,i64,i64,f64,f64
"""latino""",25.656,2.0,0,98,36.1,23.656
"""alternative""",24.337,1.0,0,93,48.5,23.337
"""reggaeton""",23.858,2.0,0,98,36.2,21.858
"""dance""",22.69,1.0,0,100,49.0,21.69
"""hip-hop""",37.759,58.0,0,99,28.9,20.241
"""soul""",19.795,0.0,0,88,61.1,19.795
"""rock""",19.001,0.0,0,96,52.5,19.001
"""reggae""",20.63,2.0,0,98,39.2,18.63
"""pop""",47.576,66.0,0,100,18.2,18.424


### Genre Popularity Is Not Uniformly Distributed

Mean and median popularity differ substantially for several genres. For example, `latino` has a mean popularity of 25.7 but a median of only 2, while `hip-hop` has a mean of 37.8 and a median of 58.

This indicates that genre-level popularity can have very different distributions, with means sometimes strongly influenced by a smaller number of highly popular tracks. Mean popularity alone therefore does not fully describe a genre.

In [14]:
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms"
]

audio_features

['danceability',
 'energy',
 'loudness',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'duration_ms']

In [15]:
correlations = [
    pl.corr(feature, "popularity").alias(feature)
    for feature in audio_features
]

df.select(correlations)

danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.035448,0.001056,0.050423,-0.044927,-0.025472,-0.095139,-0.005387,-0.040534,0.013205,-0.007101


### Audio Features and Popularity

At the overall dataset level, individual audio features show very weak linear correlations with popularity. The strongest relationship is with instrumentalness (-0.095), while danceability (0.035), energy (0.001), and tempo (0.013) are close to zero.

This suggests that no single audio characteristic has a strong linear relationship with popularity across the full dataset. Since genre shows substantial differences in popularity, the next step is to examine these relationships within genres.

In [16]:
genres = df["track_genre"].unique().sort().to_list()

In [17]:
genre_correlations = []

for genre in genres:
    genre_df = df.filter(pl.col("track_genre") == genre)

    correlations = genre_df.select(
        [
            pl.corr(feature, "popularity").alias(feature)
            for feature in audio_features
        ]
    ).row(0)

    genre_correlations.append(
        {
            "track_genre": genre,
            **dict(zip(audio_features, correlations))
        }
    )

genre_correlations = pl.DataFrame(genre_correlations)

genre_correlations

track_genre,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""acoustic""",0.050957,-0.195199,-0.148366,-0.097154,0.139507,0.00365,-0.063267,-0.190842,-0.006927,0.007007
"""afrobeat""",-0.025081,-0.048472,0.073897,0.052635,0.03895,-0.100493,0.001669,-0.0941,0.003851,-0.094384
"""alt-rock""",0.010425,-0.081982,-0.00498,-0.041461,-0.004836,-0.022442,0.038823,-0.092123,-0.050776,0.039277
"""alternative""",-0.005142,0.025122,0.039509,-0.11101,-0.079175,-0.01548,0.005576,-0.107559,-0.027852,0.060967
"""ambient""",0.038649,0.046514,0.079838,-0.041939,-0.009911,-0.124447,-0.075704,0.023383,0.016252,0.005368
…,…,…,…,…,…,…,…,…,…,…
"""techno""",0.016804,-0.08957,-0.074235,-0.020589,-0.025423,0.057282,-0.01239,-0.096638,-0.014887,-0.018158
"""trance""",-0.110488,-0.002299,0.03336,-0.062299,0.036999,-0.082941,0.001561,-0.050534,-0.013463,-0.108088
"""trip-hop""",-0.013264,-0.085389,-0.056031,-0.128311,0.057673,0.176001,-0.080946,-0.19738,-0.010255,-0.005556


In [18]:
genre_correlations.sort(
    "danceability",
    descending=True
).head(5)

track_genre,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""comedy""",0.373612,-0.112301,0.19917,-0.432883,-0.48174,0.057419,-0.40482,0.127453,0.172139,-0.013218
"""funk""",0.329847,0.044311,0.234831,0.328492,0.025992,-0.043323,0.019756,0.032855,0.099439,0.131789
"""classical""",0.211204,0.411349,0.325962,0.147799,-0.329199,-0.27573,0.035338,0.146122,0.098149,0.118388
"""garage""",0.204159,-0.2307,-0.17421,-0.124989,0.116081,0.081076,-0.126017,0.011816,-0.128617,0.042021
"""world-music""",0.196738,0.240082,0.455718,-0.096018,-0.362459,-0.479477,0.005642,0.006904,0.198512,0.043612


In [19]:
genre_correlations.sort(
    "danceability"
).head(5)

track_genre,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""sleep""",-0.348524,-0.191624,-0.40614,-0.056993,0.050342,0.320022,-0.142324,-0.285175,-0.147506,0.01625
"""hip-hop""",-0.275018,0.068263,-0.098841,0.036124,0.279801,0.071569,-0.009399,-0.022076,0.037161,0.110589
"""pagode""",-0.262188,-0.015989,-0.14295,-0.074344,0.33392,-0.009456,0.201145,0.109549,0.061011,0.123965
"""iranian""",-0.255124,-0.245854,-0.312278,-0.137468,0.144599,0.217693,-0.102048,-0.165539,-0.147327,0.124903
"""breakbeat""",-0.248069,0.059177,0.069459,-0.031016,-0.00866,0.006208,0.049226,-0.15344,0.003158,-0.06348


In [20]:
genre_correlations.sort(
    "energy",
    descending=True
).head(5)

track_genre,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""classical""",0.211204,0.411349,0.325962,0.147799,-0.329199,-0.27573,0.035338,0.146122,0.098149,0.118388
"""j-pop""",0.175203,0.401917,0.355515,0.159879,-0.376389,-0.071346,0.139326,0.23871,0.086307,0.137999
"""k-pop""",-0.081089,0.24964,0.358259,-0.045709,-0.236679,-0.011876,0.061395,0.064628,0.075129,-0.114036
"""world-music""",0.196738,0.240082,0.455718,-0.096018,-0.362459,-0.479477,0.005642,0.006904,0.198512,0.043612
"""new-age""",0.130369,0.239664,0.186334,-0.054783,-0.211227,-0.08145,-0.064565,0.072857,0.015152,-0.114806


In [21]:
genre_correlations.sort(
    "energy"
).head(5)

track_genre,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""iranian""",-0.255124,-0.245854,-0.312278,-0.137468,0.144599,0.217693,-0.102048,-0.165539,-0.147327,0.124903
"""emo""",0.15269,-0.241991,-0.135322,0.057533,0.132451,0.046268,-0.104339,-0.155771,-0.000092,-0.231731
"""study""",0.018901,-0.235063,-0.233128,-0.012824,0.143656,0.078586,-0.014665,-0.070676,-0.072803,-0.008367
"""garage""",0.204159,-0.2307,-0.17421,-0.124989,0.116081,0.081076,-0.126017,0.011816,-0.128617,0.042021
"""children""",0.160343,-0.228609,-0.143963,-0.087907,0.09343,-0.083177,-0.048716,0.067654,-0.009894,-0.149265


### Genre-Specific Relationships

The relationship between audio features and popularity varies considerably across genres. For example, energy shows a positive correlation with popularity in classical (r = 0.411) and J-pop (r = 0.402), but a negative correlation in Iranian (r = -0.246) and emo (r = -0.242) music.

This variation helps explain why the overall correlation between energy and popularity is almost zero.

In [22]:
genre_correlations.sort(
    "loudness",
    descending=True
).head(5)

track_genre,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""world-music""",0.196738,0.240082,0.455718,-0.096018,-0.362459,-0.479477,0.005642,0.006904,0.198512,0.043612
"""k-pop""",-0.081089,0.24964,0.358259,-0.045709,-0.236679,-0.011876,0.061395,0.064628,0.075129,-0.114036
"""j-pop""",0.175203,0.401917,0.355515,0.159879,-0.376389,-0.071346,0.139326,0.23871,0.086307,0.137999
"""classical""",0.211204,0.411349,0.325962,0.147799,-0.329199,-0.27573,0.035338,0.146122,0.098149,0.118388
"""dub""",-0.224834,0.156653,0.295784,-0.030394,-0.022314,-0.153091,-0.033214,-0.246872,0.114979,-0.099531


In [23]:
genre_correlations.sort(
    "loudness"
).head(5)

track_genre,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""sleep""",-0.348524,-0.191624,-0.40614,-0.056993,0.050342,0.320022,-0.142324,-0.285175,-0.147506,0.01625
"""rockabilly""",-0.202679,-0.21957,-0.313527,-0.156371,0.029883,0.046966,-0.078095,-0.180364,0.007205,0.036954
"""iranian""",-0.255124,-0.245854,-0.312278,-0.137468,0.144599,0.217693,-0.102048,-0.165539,-0.147327,0.124903
"""disco""",0.08335,-0.163217,-0.263525,-0.073575,-0.00757,0.010265,-0.094948,-0.00271,-0.021402,0.03854
"""study""",0.018901,-0.235063,-0.233128,-0.012824,0.143656,0.078586,-0.014665,-0.070676,-0.072803,-0.008367


In [24]:
correlation_long = genre_correlations.unpivot(
    index="track_genre",
    variable_name="feature",
    value_name="correlation"
)

correlation_long

track_genre,feature,correlation
str,str,f64
"""acoustic""","""danceability""",0.050957
"""afrobeat""","""danceability""",-0.025081
"""alt-rock""","""danceability""",0.010425
"""alternative""","""danceability""",-0.005142
"""ambient""","""danceability""",0.038649
…,…,…
"""techno""","""duration_ms""",-0.018158
"""trance""","""duration_ms""",-0.108088
"""trip-hop""","""duration_ms""",-0.005556


In [25]:
correlation_long.with_columns(
    pl.col("correlation").abs().alias("abs_correlation")
).sort(
    "abs_correlation",
    descending=True
).head(15)

track_genre,feature,correlation,abs_correlation
str,str,f64,f64
"""comedy""","""acousticness""",-0.48174,0.48174
"""world-music""","""instrumentalness""",-0.479477,0.479477
"""world-music""","""loudness""",0.455718,0.455718
"""comedy""","""speechiness""",-0.432883,0.432883
"""classical""","""energy""",0.411349,0.411349
…,…,…,…
"""blues""","""acousticness""",-0.372137,0.372137
"""world-music""","""acousticness""",-0.362459,0.362459
"""k-pop""","""loudness""",0.358259,0.358259


## Key Findings from Initial EDA

- The dataset contains **114,000 tracks across 114 genres**, with exactly **1,000 tracks per genre**.
- Tracks are grouped by genre in the original dataset, so the raw row order is not random.
- Popularity is concentrated in the lower and middle ranges, with nearly **90% of tracks scoring 60 or below**.
- The prevalence of zero-popularity tracks varies substantially across genres, reaching over **60% in some genres**.
- Genre-level zero-popularity percentage has a **moderate negative correlation (-0.429)** with mean popularity.
- Individual audio features have **very weak overall linear correlations** with popularity.
- However, these relationships can change considerably when examined **within individual genres**.
- The strongest genre-specific relationships observed were between **acousticness and popularity in comedy (-0.482)**, **instrumentalness and popularity in world-music (-0.479)**, and **loudness and popularity in world-music (+0.456)**.

### Direction for Further Analysis

The initial EDA suggests that popularity cannot be explained well by a single audio feature across all genres. Further analysis will investigate these relationships using **DuckDB, PostgreSQL, and additional statistical analysis**.

## Analytical SQL with DuckDB

In [26]:
import duckdb

In [27]:
result = duckdb.sql("""
    SELECT
        track_genre,
        COUNT(*) AS tracks,
        ROUND(AVG(popularity), 2) AS avg_popularity
    FROM '../data/spotify-tracks-dataset-detailed.csv'
    GROUP BY track_genre
    ORDER BY avg_popularity DESC
    LIMIT 10
""")

result

┌─────────────┬────────┬────────────────┐
│ track_genre │ tracks │ avg_popularity │
│   varchar   │ int64  │     double     │
├─────────────┼────────┼────────────────┤
│ pop-film    │   1000 │          59.28 │
│ k-pop       │   1000 │           56.9 │
│ chill       │   1000 │          53.65 │
│ sad         │   1000 │          52.38 │
│ grunge      │   1000 │          49.59 │
│ indian      │   1000 │          49.54 │
│ anime       │   1000 │          48.77 │
│ emo         │   1000 │          48.13 │
│ sertanejo   │   1000 │          47.87 │
│ pop         │   1000 │          47.58 │
└─────────────┴────────┴────────────────┘
  10 rows                     3 columns

### DuckDB: Genre Popularity

DuckDB allows us to query the CSV directly using SQL without first loading the entire dataset into a DataFrame.

The results show substantial differences in average popularity across genres, with pop-film having the highest average popularity (59.28) and the dataset containing 1,000 tracks per genre.

In [28]:
duckdb.sql("""
    SELECT
        explicit,
        COUNT(*) AS tracks,
        ROUND(AVG(popularity), 2) AS avg_popularity,
        ROUND(MEDIAN(popularity), 2) AS median_popularity
    FROM '../data/spotify-tracks-dataset-detailed.csv'
    GROUP BY explicit
    ORDER BY explicit
""")

┌──────────┬────────┬────────────────┬───────────────────┐
│ explicit │ tracks │ avg_popularity │ median_popularity │
│ boolean  │ int64  │     double     │      double       │
├──────────┼────────┼────────────────┼───────────────────┤
│ false    │ 104253 │          32.94 │              34.0 │
│ true     │   9747 │          36.45 │              37.0 │
└──────────┴────────┴────────────────┴───────────────────┘

### Explicit Content and Popularity

Explicit tracks have a higher average popularity (36.45) than non-explicit tracks (32.94), with a similar difference in median popularity (37 vs. 34).

This is an association within the dataset, not evidence that explicit content causes higher popularity. Genre and other characteristics may also influence popularity.

In [29]:
duckdb.sql("""
    SELECT
        artists,
        unnest(string_split(artists, ';')) AS artist
    FROM '../data/spotify-tracks-dataset-detailed.csv'
    LIMIT 10
""")

┌──────────────────────────────────────┬────────────────────┐
│               artists                │       artist       │
│               varchar                │      varchar       │
├──────────────────────────────────────┼────────────────────┤
│ Gen Hoshino                          │ Gen Hoshino        │
│ Ben Woodward                         │ Ben Woodward       │
│ Ingrid Michaelson;ZAYN               │ Ingrid Michaelson  │
│ Ingrid Michaelson;ZAYN               │ ZAYN               │
│ Kina Grannis                         │ Kina Grannis       │
│ Chord Overstreet                     │ Chord Overstreet   │
│ Tyrone Wells                         │ Tyrone Wells       │
│ A Great Big World;Christina Aguilera │ A Great Big World  │
│ A Great Big World;Christina Aguilera │ Christina Aguilera │
│ Jason Mraz                           │ Jason Mraz         │
└──────────────────────────────────────┴────────────────────┘
  10 rows                                         2 columns

In [30]:
duckdb.sql("""
    SELECT
        artist,
        COUNT(*) AS tracks,
        ROUND(AVG(popularity), 2) AS avg_popularity
    FROM (
        SELECT
            unnest(string_split(artists, ';')) AS artist,
            popularity
        FROM '../data/spotify-tracks-dataset-detailed.csv'
    )
    GROUP BY artist
    HAVING COUNT(*) >= 10
    ORDER BY avg_popularity DESC
    LIMIT 10
""")

┌───────────────────┬────────┬────────────────┐
│      artist       │ tracks │ avg_popularity │
│      varchar      │ int64  │     double     │
├───────────────────┼────────┼────────────────┤
│ Måneskin          │     12 │          83.67 │
│ Lil Nas X         │     11 │          83.45 │
│ ROSALÍA           │     11 │          77.09 │
│ Radiohead         │     20 │          76.65 │
│ Regard            │     10 │           76.1 │
│ The Neighbourhood │     60 │          75.58 │
│ Lost Frequencies  │     10 │           74.5 │
│ System Of A Down  │     17 │          74.47 │
│ Limp Bizkit       │     12 │          74.33 │
│ Frank Ocean       │     23 │          73.96 │
└───────────────────┴────────┴────────────────┘
  10 rows                           3 columns

In [31]:
duckdb.sql("""
    SELECT
        artist,
        COUNT(*) AS tracks
    FROM (
        SELECT
            unnest(string_split(artists, ';')) AS artist
        FROM '../data/spotify-tracks-dataset-detailed.csv'
    )
    GROUP BY artist
    ORDER BY tracks DESC
    LIMIT 10
""")

┌─────────────────────────┬────────┐
│         artist          │ tracks │
│         varchar         │ int64  │
├─────────────────────────┼────────┤
│ J Balvin                │    510 │
│ Bad Bunny               │    416 │
│ Daddy Yankee            │    375 │
│ Wolfgang Amadeus Mozart │    354 │
│ Feid                    │    348 │
│ George Jones            │    343 │
│ Pritam                  │    333 │
│ Arijit Singh            │    305 │
│ ILLENIUM                │    298 │
│ The Beatles             │    280 │
└─────────────────────────┴────────┘
  10 rows                2 columns

## DuckDB Analysis

DuckDB was used to perform analytical SQL directly on the CSV dataset.

Key observations:

- `pop-film` had the highest average popularity among the 114 genres at **59.28**.
- Explicit tracks had a higher average popularity than non-explicit tracks (**36.45 vs. 32.94**).
- After separating collaborations into individual artists, **Måneskin** had the highest average popularity among artists with at least 10 tracks (**83.67**).
- **J Balvin** appeared on the highest number of tracks in the dataset (**510**), followed by Bad Bunny (**416**) and Daddy Yankee (**375**).

These queries demonstrated how DuckDB can handle aggregation, filtering, grouping, and nested transformations directly over the CSV.

# PostgreSQL Analysis

### Track ID Duplication

A database integrity check revealed that the dataset contains **114,000 records but only 89,741 unique track IDs**.

Further investigation showed that repeated track IDs are often associated with different genre labels. For example, one track appeared under nine different genre labels.

Therefore, the dataset should be treated as **genre-labelled records rather than 114,000 unique Spotify tracks**. This distinction is important when interpreting track-level statistics.